In [1]:
# Import the module
import importlib
import IsovizPy as ja
import pandas as pd
import gffutils

In [2]:
# After making changes to IsovizPy, reload it (mostly for testing purposes)
importlib.reload(ja)

<module 'IsovizPy' from '/gpfs/commons/home/kisaev/Leaflet-private/src/visualization/IsovizPy.py'>

In [3]:
# Load data and create the database (note this may take 1-2 minutes - do this once!)
gtf_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/genes/genes.gtf"  
fasta_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/genome_files/gencode.vM19/fasta/genome.fa"

In [4]:
# Path to the database file created previously
db_path = "/gpfs/commons/home/kisaev/Leaflet-analysis/TabulaSenis/LeafletFA_analysis/gencode.vM19"

# Load the database
db = gffutils.FeatureDB(db_path, keep_order=True)

In [5]:
# Define the junctions and target format for conversion
junctions = [
    ("chr6", 48838362, 48840360, "-"),
    ("chr6", 48838365, 48840995, "-"),
    ("chr6", 48838365, 48840360, "-"),
    ("chr6", 48840453, 48840995, "-"),
    ("chr6", 48840448, 48840995, "-")
]

# If you have usage ratios, you can also specify those here otherwise, make a column in data called "usage_ratio" and make all values 0 to turn off this functionality 
usage_ratios = [0, 0, 0, 0, 0]

# Adjust the 'usage_ratio' values to be rounded to the nearest whole number for percentage format
data = {
    'junction_id': [f"{chrom}_{start}_{end}_{strand}" for chrom, start, end, strand in junctions],
    'Cluster': [55951] * len(junctions),  # Assign all to a single cluster for simplicity
    'usage_ratio': [round(ratio) for ratio in usage_ratios]  # Round to the nearest whole number
}

In [6]:
splice_adata_var = pd.DataFrame(data)
print(splice_adata_var)

                junction_id  Cluster  usage_ratio
0  chr6_48838362_48840360_-    55951            0
1  chr6_48838365_48840995_-    55951            0
2  chr6_48838365_48840360_-    55951            0
3  chr6_48840453_48840995_-    55951            0
4  chr6_48840448_48840995_-    55951            0


In [7]:
# Convert junction_ids to format that can easily be overlapped with transcript coordinates stored in our database 
splice_junctions = ja.convert_junction_ids(splice_adata_var)
print(splice_junctions)

[{'chrom': 'chr6', 'start': 48838362, 'end': 48840360, 'name': 'junction_1', 'strand': '-', 'usage_ratio': 0}, {'chrom': 'chr6', 'start': 48838365, 'end': 48840995, 'name': 'junction_2', 'strand': '-', 'usage_ratio': 0}, {'chrom': 'chr6', 'start': 48838365, 'end': 48840360, 'name': 'junction_3', 'strand': '-', 'usage_ratio': 0}, {'chrom': 'chr6', 'start': 48840453, 'end': 48840995, 'name': 'junction_4', 'strand': '-', 'usage_ratio': 0}, {'chrom': 'chr6', 'start': 48840448, 'end': 48840995, 'name': 'junction_5', 'strand': '-', 'usage_ratio': 0}]


In [8]:
ja.check_splice_site_motifs(fasta_file, splice_junctions)

[{'chrom': 'chr6',
  'start': 48838362,
  'end': 48840360,
  'name': 'junction_1',
  'strand': '-',
  'usage_ratio': 0,
  'splice_motif': 'non-canonical',
  'donor_seq': 'CT',
  'acceptor_seq': 'AC'},
 {'chrom': 'chr6',
  'start': 48838365,
  'end': 48840995,
  'name': 'junction_2',
  'strand': '-',
  'usage_ratio': 0,
  'splice_motif': 'non-canonical',
  'donor_seq': 'CT',
  'acceptor_seq': 'AC'},
 {'chrom': 'chr6',
  'start': 48838365,
  'end': 48840360,
  'name': 'junction_3',
  'strand': '-',
  'usage_ratio': 0,
  'splice_motif': 'non-canonical',
  'donor_seq': 'CT',
  'acceptor_seq': 'AC'},
 {'chrom': 'chr6',
  'start': 48840453,
  'end': 48840995,
  'name': 'junction_4',
  'strand': '-',
  'usage_ratio': 0,
  'splice_motif': 'non-canonical',
  'donor_seq': 'CT',
  'acceptor_seq': 'AC'},
 {'chrom': 'chr6',
  'start': 48840448,
  'end': 48840995,
  'name': 'junction_5',
  'strand': '-',
  'usage_ratio': 0,
  'splice_motif': 'non-canonical',
  'donor_seq': 'CT',
  'acceptor_seq': 'A

In [ ]:
# Check junction annotations
junction_annotation_results = ja.check_junction_annotation(splice_junctions, db)
print(junction_annotation_results)

# Extract unique transcript IDs from junction_labels
unique_transcripts = list({transcript for label in junction_annotation_results for transcript in label['transcripts']})
print(unique_transcripts)

In [ ]:
# Fetch transcript exon coordinates and determine plot boundaries
transcript_data = ja.fetch_transcripts_and_annotations(db, unique_transcripts)

In [ ]:
# Get start and end coordinates for our plot 
region_start, region_end = ja.determine_region_boundaries(splice_junctions)

In [ ]:
# Optionally - confirm whether your splice junctions fully annotate to known exons on 5' or 3' side! 
ja.check_junction_annotation(splice_junctions, db)

In [ ]:
# Plot the annotations and splice junctions
ja.plot_exons_and_junctions(db, transcript_data, splice_junctions, region_start-500, region_end-700, base_width=10, trans_height=0.5, show_usage=False, show_junc_lines=True)

In [ ]:
# Plot the annotations and splice junctions
ja.plot_exons_and_junctions(db, transcript_data, splice_junctions, region_start-500, region_end-700, base_width=9, trans_height=0.5, show_usage=False, show_junc_lines=True)

In [ ]:
# Plot the annotations and splice junctions
ja.plot_exons_and_junctions(db, transcript_data, splice_junctions, region_start-500, region_end-700, base_width=10, trans_height=0.8, show_usage=True, colorbar_pad=0.3 ,show_junc_lines=False)

In [ ]:
# Plot the annotations and splice junctions
ja.plot_exons_and_junctions(db, transcript_data, splice_junctions, region_start-500, region_end-700, base_width=10, trans_height=0.8, show_usage=True, colorbar_pad=0.3 ,show_junc_lines=True)

#### If you just want to visualize all of TREM2 isoforms without any specific splice junctions you can do the following:

In [ ]:
gene_name = "TREM2"

In [ ]:
transcript_ids = ja.fetch_transcripts_for_gene(db, gene_name)

# Step 2: Fetch transcript annotations
transcript_data = ja.fetch_transcripts_and_annotations(db, transcript_ids)
    
# Step 3: Determine region boundaries
region_start, region_end = ja.determine_region_boundaries_from_transcripts(transcript_data)

In [ ]:
# Step 4: Plot isoforms
ja.plot_isoforms(db, transcript_data, region_start-500, region_end+200, base_width=10, trans_height=0.3)